**Building an RAG Pipeline Using PDF Chunking and Retrieval**

Build a a Retrieval-Augmented-Generation Pipeline using various tools in langchain library to process PDF documents , convert them into vectorized chunks and retrieve relevant information using semantic search .
Learnings  - STrengthen understanding on-  real world content


*   Document chunking
*   Embedding
*   Vector search workflow


Relative path for loading the pdfs-
<!--
/content/sample_data/rag_pdfs/beyond-the-copilot-scaling-the-agentic-product-development-life-cycle.pdf

/content/sample_data/rag_pdfs/one-year-of-agentic-ai-six-lessons-from-the-people-doing-the-work_vf.pdf -->


/content/sample_data/cep_data_monetization_problem_11.pdf

/content/sample_data/cep_data_monetization_problem_2.pdf

/content/sample_data/Data_Monetisation.pdf

# Step 1: Install dependencies including pdf and text loaders

In [2]:
# Import the libraries , all of the tools for agent building
!pip install langchain openai PyPDF2 faiss-cpu tiktoken langchain-openai langchain-classic pypdf
!pip install langchain_community
!pip install langchain_text_splitters
!pip install langchain_classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires req

# Step 2: Import dependencies

In [3]:
import os
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter# for chunking
# vectorstores is for storage purposes  # computers only understand numbers hence vectorize the text, called embeddings store in vector db
# FAISS is one of the classes helping to store the embeddings in a vector db
from langchain_classic.chains import RetrievalQA  # helping to load some documets , load some embeddings
#update

from langchain_community.vectorstores import FAISS # This is for
from langchain_community.document_loaders import TextLoader
# import the libraries for PDFs
from langchain_community.document_loaders import PyPDFLoader
from PyPDF2 import PdfReader

/tmp/ipykernel_1111/1653098350.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader


# Step 3: Set Azure OpenAI credentials

In [7]:
os.environ["AZURE_OPENAI_API_KEY"] = ""

# Step 4: Load and chunk your custom FAQ document

Preserves semantic boundaries, making chunks safer for embeddings, search, and RAG workflows



In [4]:
from langchain_core import documents
loader = DirectoryLoader(
    path='/content/sample_data/',
    glob='*.pdf',
    loader_cls=PyPDFLoader
)

documents = loader.lazy_load()

for document in documents:
    print(document.metadata)

{'producer': 'Skia/PDF m154 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'cep_data_monetization_problem_2.docx', 'source': '/content/sample_data/cep_data_monetization_problem_2.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}
{'producer': 'Skia/PDF m154 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'cep_data_monetization_problem_2.docx', 'source': '/content/sample_data/cep_data_monetization_problem_2.pdf', 'total_pages': 4, 'page': 1, 'page_label': '2'}
{'producer': 'Skia/PDF m154 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'cep_data_monetization_problem_2.docx', 'source': '/content/sample_data/cep_data_monetization_problem_2.pdf', 'total_pages': 4, 'page': 2, 'page_label': '3'}
{'producer': 'Skia/PDF m154 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'cep_data_monetization_problem_2.docx', 'source': '/content/sample_data/cep_data_monetization_problem_2.pdf', 'total_pages': 4

Step 5: Split into chucks all of the loaded documents

In [8]:
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)# how do we define the chunk size ? overlap is needed to retain context across chunks

# Re-load the documents because the generator was consumed in the previous cell.
documents = loader.lazy_load()
# Convert the documents generator to a list.
documents_list = list(documents)

docs = text_splitter.split_documents(documents_list)

# Check if the loader is initialized correctly
print(f'Loaded {len(documents_list)} documents')
print(f'Split into {len(docs)} chunks')

Loaded 12 documents
Split into 12 chunks


# Step 5: Create vectorstore using Azure embeddings


In [10]:
# Embed documnet chunks and store them in an FAISS vector store
embedddings = AzureOpenAIEmbeddings(
    azure_endpoint="https://openai-api-management-gw.azure-api.net",
    deployment="text-embedding-ada-002",
    chunk_size=500,
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version="2023-05-15",
    model="text-embedding-ada-002",
    chunk_overlap=100
)
vectorstore = FAISS.from_documents(docs, embedddings)


# Step 6: Initialize the Azure OpenAI LLM

In [12]:
# init the azure openai llm for deterministic output , set temperature = 0
llm = AzureChatOpenAI(
    azure_endpoint="https://openai-api-management-gw.azure-api.net",
    api_version="2025-01-01-preview",
    deployment_name="gpt-5-mini"

)
#

\# Step 7: Create the RAG chain

In [13]:
qa_chain= RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(),

    return_source_documents=True
)

NameError: name 'llm' is not defined

# Step 8: Ask a question

In [15]:
query1= "What examples are existing in the RAG store with respect to Data monetization ? "
result1 = qa_chain.invoke("query":query1)

query2= "What are the different types of Data monetization described in the documents ? "
result2 = qa_chain.invoke("query2":query2)

query3= "What do the customers of Medtronic gain as per the example of Data monetization ? "
result3 = qa_chain.invoke("query2":query2)

query4= "What careful considerations must be made during dmonetization opportunities ? "
result4 = qa_chain.invoke("query2":query2)

# Step 9: Print results

In [14]:
# Print all of the above results
print(result1)
print(result2)
print(result3)
print(result4)

NameError: name 'result1' is not defined